In [20]:
import sys, os

# Always make project root importable
sys.path.append(os.path.abspath(".."))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv('/Users/zainulislam/Documents/Tiktok Listings /Data Science ENV/data-science-env/data/train.csv')

# Separate features and target
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

# Split immediately after loading
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [4]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Example: categorical columns
cat_cols = ['Neighborhood', 'HouseStyle']
num_cols = ['LotArea', 'OverallQual', 'YearBuilt']

# Persistent encoders/scalers
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
scaler = StandardScaler()

# Fit on training data
X_train_cat = encoder.fit(X_train[cat_cols]).transform(X_train[cat_cols])
X_train_num = scaler.fit(X_train[num_cols]).transform(X_train[num_cols])

# Combine processed features
import numpy as np
X_train_processed = np.hstack([X_train_num, X_train_cat])

In [5]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=42)
model.fit(X_train_processed, y_train)

RandomForestRegressor(random_state=42)

In [6]:
# Process test data using the same encoders/scalers
X_test_cat = encoder.transform(X_test[cat_cols])
X_test_num = scaler.transform(X_test[num_cols])
X_test_processed = np.hstack([X_test_num, X_test_cat])

# Predictions
y_pred = model.predict(X_test_processed)

# Evaluation
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {rmse:.4f}")

RMSE: 38268.9536


In [9]:
# Load new data
inference_df = pd.read_csv('/Users/zainulislam/Documents/Tiktok Listings /Data Science ENV/data-science-env/data/test.csv')

# Apply same preprocessing
X_inf_cat = encoder.transform(inference_df[cat_cols])
X_inf_num = scaler.transform(inference_df[num_cols])
X_inf_processed = np.hstack([X_inf_num, X_inf_cat])

# Predict
predictions = model.predict(X_inf_processed)
predictions

array([135022.55, 169472.5 , 176150.5 , ..., 132780.  , 158237.15,
       214728.  ])

In [4]:
import os
os.makedirs("models", exist_ok=True)

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv('../data/train.csv')

# Separate features and target
X = df.drop(columns=['SalePrice'])  # Replace 'SalePrice' with your target column
y = df['SalePrice']

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np

# Define categorical and numerical columns
cat_cols = ['Neighborhood', 'HouseStyle']  # Replace with your categorical columns
num_cols = ['LotArea', 'OverallQual', 'YearBuilt']  # Replace with your numerical columns

# Create persistent encoder and scaler
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
scaler = StandardScaler()

# Fit and transform training data
X_train_cat = encoder.fit(X_train[cat_cols]).transform(X_train[cat_cols])
X_train_num = scaler.fit(X_train[num_cols]).transform(X_train[num_cols])

# Combine processed numerical and categorical features
X_train_processed = np.hstack([X_train_num, X_train_cat])

In [8]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=42)
model.fit(X_train_processed, y_train)

RandomForestRegressor(random_state=42)

In [9]:
import os

# Create models folder if it doesn't exist
os.makedirs("models", exist_ok=True)

In [10]:
import joblib

# Save trained model
joblib.dump(model, "models/model.joblib")

# Save encoder
joblib.dump(encoder, "models/encoder.joblib")

# Save scaler
joblib.dump(scaler, "models/scaler.joblib")

['models/scaler.joblib']

In [12]:
print(os.listdir("models"))

['scaler.joblib', 'model.joblib', 'encoder.joblib']


In [13]:
# Load objects
loaded_model = joblib.load("models/model.joblib")
loaded_encoder = joblib.load("models/encoder.joblib")
loaded_scaler = joblib.load("models/scaler.joblib")

# Example: preprocess test data
X_test_cat = loaded_encoder.transform(X_test[cat_cols])
X_test_num = loaded_scaler.transform(X_test[num_cols])
X_test_processed = np.hstack([X_test_num, X_test_cat])

# Predict
predictions = loaded_model.predict(X_test_processed)
print(predictions[:10])  # Show first 10 predictions

[156433.         303190.19       111989.6        127208.33333333
 334540.99        83970.         253584.         148615.
  84880.         164604.2       ]


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import joblib
import os

def build_model(data: pd.DataFrame) -> dict[str, float]:
    """
    Trains a RandomForestRegressor model on the given dataset, 
    persists the model, encoder, and scaler, and returns performance metrics.
    
    Args:
        data (pd.DataFrame): Training dataset with features and target column 'SalePrice'.
        
    Returns:
        dict: Dictionary containing model performance metrics (RMSE).
    """
    # Separate features and target
    X = data.drop(columns=['SalePrice'])
    y = data['SalePrice']
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Identify categorical and numerical columns
    cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
    num_cols = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
    
    # Create persistent encoder and scaler
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    scaler = StandardScaler()
    
    # Fit and transform training data
    X_train_cat = encoder.fit(X_train[cat_cols]).transform(X_train[cat_cols])
    X_train_num = scaler.fit(X_train[num_cols]).transform(X_train[num_cols])
    X_train_processed = np.hstack([X_train_num, X_train_cat])
    
    # Train model
    model = RandomForestRegressor(random_state=42)
    model.fit(X_train_processed, y_train)
    
    # Persist objects
    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/model.joblib")
    joblib.dump(encoder, "models/encoder.joblib")
    joblib.dump(scaler, "models/scaler.joblib")
    
    # Evaluate on test set
    X_test_cat = encoder.transform(X_test[cat_cols])
    X_test_num = scaler.transform(X_test[num_cols])
    X_test_processed = np.hstack([X_test_num, X_test_cat])
    
    y_pred = model.predict(X_test_processed)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    return {"rmse": rmse}

In [15]:
def make_predictions(input_data: pd.DataFrame) -> np.ndarray:
    """
    Loads persisted model and preprocessing objects, 
    preprocesses the input data, and predicts house prices.
    
    Args:
        input_data (pd.DataFrame): New dataset to predict.
        
    Returns:
        np.ndarray: Predicted house prices.
    """
    # Load persisted objects
    model = joblib.load("models/model.joblib")
    encoder = joblib.load("models/encoder.joblib")
    scaler = joblib.load("models/scaler.joblib")
    
    # Identify categorical and numerical columns (must match training)
    cat_cols = input_data.select_dtypes(include=['object']).columns.tolist()
    num_cols = input_data.select_dtypes(include=['int64','float64']).columns.tolist()
    
    # Transform data
    X_cat = encoder.transform(input_data[cat_cols])
    X_num = scaler.transform(input_data[num_cols])
    X_processed = np.hstack([X_num, X_cat])
    
    # Predict
    predictions = model.predict(X_processed)
    return predictions

In [16]:
# Build and train model
import pandas as pd
training_df = pd.read_csv("../data/train.csv")
metrics = build_model(training_df)
print(metrics)

# Make predictions on new data
test_df = pd.read_csv("../data/test.csv")
preds = make_predictions(test_df)
print(preds[:10])

{'rmse': 29248.14299180536}
[129690.   156320.5  178953.   190559.   206001.66 183511.4  170579.75
 177052.92 182078.19 122993.93]


In [17]:
# Model Building
import pandas as pd
from house_prices.train import build_model

training_df = pd.read_csv("../data/train.csv")
metrics = build_model(training_df)
print(metrics)

# Model Inference
from house_prices.inference import make_predictions

test_df = pd.read_csv("../data/test.csv")
predictions = make_predictions(test_df)
print(predictions[:10])

ModuleNotFoundError: No module named 'house_prices'

In [18]:
# Model Building
import pandas as pd
from house_prices.train import build_model

training_df = pd.read_csv("../data/train.csv")
metrics = build_model(training_df)
print(metrics)

# Model Inference
from house_prices.inference import make_predictions

test_df = pd.read_csv("../data/test.csv")
predictions = make_predictions(test_df)
print(predictions[:10])

ModuleNotFoundError: No module named 'house_prices'

In [19]:
!ls ..

README.md        data             models           requirements.txt
Untitled.ipynb   house_prices     notebooks


In [21]:
from house_prices.train import build_model
from house_prices.inference import make_predictions

ImportError: cannot import name 'build_model' from 'house_prices.train' (/Users/zainulislam/Documents/Tiktok Listings /Data Science ENV/data-science-env/house_prices/train.py)